In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient


# Load HF Token
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found. Please set it in .env")


# List of models to process
MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "MiniMaxAI/MiniMax-M2",
    "meta-llama/Llama-3.1-8B-Instruct",
    "openai/gpt-oss-20b",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
]


def load_text_file(path):
    """Load a UTF-8 encoded text file from disk, 
       validate its existence and minimum length, and 
       return the file content as a string for downstream LLM processing."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if len(text.strip()) < 50:
        raise ValueError("Input text too short or empty for extraction.")

    return text


def run_llm(model_name, prompt):
    """Send a prompt to the specified LLM model via Hugging Face Inference API 
       and return the generated response text for FIR extraction."""
    client = InferenceClient(model_name, token=HF_TOKEN)
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": "You are a strict TOON document extraction engine. Follow schema exactly."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=8192,
        temperature=0.1,
    )

    return response.choices[0].message["content"]


def extract_information(text, model_name):
    """Construct a strict TOON-compliant FIR extraction prompt using predefined schema and disambiguation rules,
       inject the parsed FIR text, and invoke the LLM to generate structured output."""
    instruction = """
You are a strict FIR document information extraction engine.

TASK:
Extract structured information from the following FIR document text and output it ONLY in the exact TOON (Token-Oriented Object Notation) format defined below.

RULES (MANDATORY):
- Do NOT summarize.
- Do NOT explain.
- Do NOT output JSON.
- Do NOT add commentary.
- Do NOT change Tamil text.
- Correct only obvious English OCR mistakes.
- Preserve FIR narrative FULLY without shortening.
- FIR_Contents MUST contain the complete bilingual complaint text.
- If any field is missing, write: Not Available
- Maintain exact field order.
- Maintain exact TOON structure.
- Do NOT invent any values.
- Values MUST NOT contain standalone empty lines.
- Do NOT shift values to next fields.
- Very long text must remain inside FIR_Contents ONLY.

========================
TOON FORMAT (FIXED)
========================

Document_Type: First Information Report (FIR)

Fields{
FIR_Serial_Number,
FIR_Number,
Year,
District,
Police_Station,
Date_of_Information,
Time_of_Information,
Type_of_Information,
Sections_of_Law,
Occurrence_Day,
Occurrence_Date_From,
Occurrence_Date_To,
Occurrence_Time_From,
Occurrence_Time_To,
Place_of_Occurrence_Distance,
Place_of_Occurrence_Direction,
Place_of_Occurrence_Address,
Complainant_Name,
Complainant_Father_or_Husband,
Complainant_Nationality,
Complainant_Occupation,
Complainant_Address,
Accused_Names,
Reason_for_Delay,
Property_Details,
Total_Property_Value,
Inquest_or_Unnatural_Death_Details,
FIR_Contents,
Action_Taken,
Court_Despatch_Date_Time,
Investigating_Officer_Name,
Investigating_Officer_Rank,
Investigating_Officer_Station
}

Values{
<value_1>,
<value_2>,
<value_3>,
<value_4>,
<value_5>,
<value_6>,
<value_7>,
<value_8>,
<value_9>,
<value_10>,
<value_11>,
<value_12>,
<value_13>,
<value_14>,
<value_15>,
<value_16>,
<value_17>,
<value_18>,
<value_19>,
<value_20>,
<value_21>,
<value_22>,
<value_23>,
<value_24>,
<value_25>,
<value_26>,
<value_27>,
<value_28>,
<value_29>,
<value_30>,
<value_31>,
<value_32>
}

========================
CRITICAL DISAMBIGUATION RULES
========================
- Printed numbers like 7405314 MUST go only into FIR_Serial_Number.
- Actual FIR numbers like "1/2017", "26/2017", "11/2017" MUST go only into FIR_Number.
- The longest Tamil+English narrative paragraph MUST go only into FIR_Contents.
- FIR_Contents MUST NOT be split across multiple values.
- FIR_Contents may exceed 2000 characters.
- Tamil text MUST remain exactly as-is.

========================
DOCUMENT CONTENT:
{text_data}

========================
OUTPUT:

"""

    prompt = instruction.replace("{text_data}", text)

    return run_llm(model_name, prompt)


def get_output_filename(model_name):
    """Generate output filename based on model name.
       Converts model path to a clean filename."""
    # Extract model identifier from path (e.g., "Qwen2.5-7B-Instruct" from "Qwen/Qwen2.5-7B-Instruct")
    model_id = model_name.split("/")[-1]
    return f"../data/llm_extracted_outputs/{model_id}_output.txt"


# Main Execution
if __name__ == "__main__":
    txt_path = "../data/output_text_Cr.No.01-2017.txt"
    
    # Create output directory
    os.makedirs("../data/llm_extracted_outputs", exist_ok=True)
    
    print("Loading text file...")
    text_data = load_text_file(txt_path)
    
    print(f"\nProcessing {len(MODELS)} models...\n")
    print("=" * 60)
    
    # Process each model
    for i, model_name in enumerate(MODELS, 1):
        try:
            print(f"\n[{i}/{len(MODELS)}] Processing: {model_name}")
            print("-" * 60)
            
            # Extract information using current model
            print(f"  ➜ Extracting information...")
            result = extract_information(text_data, model_name)
            
            # Save output with model-specific filename
            output_path = get_output_filename(model_name)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(result)
            
            print(f"  ✓ Output saved to: {output_path}")
            
        except Exception as e:
            print(f"  ✗ Error processing {model_name}: {str(e)}")
            continue
    
    print("\n" + "=" * 60)
    print("✓ All models processed successfully!")
    print("=" * 60)

Loading text file...

Processing 5 models...


[1/5] Processing: Qwen/Qwen2.5-7B-Instruct
------------------------------------------------------------
  ➜ Extracting information...
  ✓ Output saved to: ../data/llm_extracted_outputs/Qwen2.5-7B-Instruct_output.txt

[2/5] Processing: MiniMaxAI/MiniMax-M2
------------------------------------------------------------
  ➜ Extracting information...
  ✓ Output saved to: ../data/llm_extracted_outputs/MiniMax-M2_output.txt

[3/5] Processing: meta-llama/Llama-3.1-8B-Instruct
------------------------------------------------------------
  ➜ Extracting information...
  ✗ Error processing meta-llama/Llama-3.1-8B-Instruct: (Request ID: Root=1-69391388-6b0aed472ad1299c4b233777;da1b055e-0bc7-40ce-91ab-54a59f965c6f)

Bad request:

[4/5] Processing: openai/gpt-oss-20b
------------------------------------------------------------
  ➜ Extracting information...
  ✓ Output saved to: ../data/llm_extracted_outputs/gpt-oss-20b_output.txt

[5/5] Processing: deepsee